In [11]:
import pandas as pd
import ast
dir = "dataset/ecg-qa/ecgqa/ptbxl/answers_for_each_template.csv"

In [13]:
df_answers = pd.read_csv(dir)
ast.literal_eval(df_answers.iloc[0]["classes"])
classes_series = df_answers["classes"]
classes_series=classes_series.apply(ast.literal_eval)
df_answers["classes"]=classes_series
df_answers.head()


,template_id,classes
0,1,"[no, not sure, yes]"
1,2,"[no, yes]"
2,3,"[conduction disturbance, hypertrophy, myocardi..."
3,4,"[complete left bundle branch block, complete r..."
4,5,"[complete left bundle branch block, complete r..."


In [19]:
class Template_Answer:
    def __init__(self,template_id,classes):
        self.template_id = template_id
        self.classes = classes
        self.classes = tuple(classes)

    def __eq__(self,other):
        if isinstance(other,Template_Answer):
            if len(self.classes) != len(other.classes):
                return False
            for i in range(len(self.classes)):
                if not(self.classes[i] in other.classes):
                    return False
            return True
        return False
    def __hash__(self):
        list_classes = list(self.classes)
        list_classes.sort()
        list_classes = [str(i).strip().lower() for i in list_classes]
        return hash(tuple(list_classes))
    def sorted_classes(self):
        list_classes = list(self.classes)
        list_classes.sort()
        list_classes = [str(i).strip().lower() for i in list_classes]
        return tuple(list_classes)




In [36]:
ta_list = []
for i in range(len(df_answers)):
    ta_list.append(Template_Answer(df_answers.iloc[i]["template_id"],df_answers.iloc[i]["classes"]))
unique_ta_list = list(set(ta_list))
ta_dict = {}
for ta in unique_ta_list:
    ta_dict[ta] = []
for ta in ta_list:
    ta_dict[ta].append(ta)
res = []
for key,value in ta_dict.items():
    if "yes" in key.sorted_classes():
        continue
    id_list = [int(i.template_id) for i in value]

    res.append({"classes":str(list(key.sorted_classes())),"ids":str(id_list)})
df_res = pd.DataFrame(res)
df_res.to_csv("res.csv",index=False)




In [27]:
dir = "dataset/ecg-qa/ecgqa/ptbxl/template/valid"
import os
import json
files = os.listdir(dir)
res = {}
for file in files:
    with open(os.path.join(dir,file),"r") as f:
        datas = json.load(f)
    for data in datas:
        res[data["template_id"]] = data["question"]
json.dump(res,open("question.json","w"))





In [33]:
res[56]

'What numeric features of the recent tracing are still abnormal compared to the previous one?'